# Sector ETF Signal Evaluation: MA Crossover, RSI and Donchian

## 1. Research Question and Project Overview

**Question:** Can simple technical signals generate superior risk-adjusted returns to the S&P 500 when applied systematically to US sector ETFs?

| # | Signal | Type |
|---|---|---|
| 0 | Moving Average Crossover | Trend-following |
| 1 | RSI | Mean-reversion |
| 2 | Donchian Channel Breakout | Breakout / trend-following |

Each signal is tested on all 10 SPDR/iShares sector ETFs. The ETF with the highest in-sample Sortino ratio is assigned to that signal. IS-optimal parameters are then applied unchanged to two out-of-sample windows.

**Periods:**
- In-sample (IS): 2010–2019 — parameter optimisation only
- OOS1: 2020–2025 — forward validation
- OOS2: 2000–2009 — backward / pre-sample stress-test

**Benchmark:** S&P 500 buy-and-hold (`^GSPC`)

## 2. Imports and Global Settings

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import pathlib
%matplotlib inline

import module
importlib.reload(module)

required = [
    'ma_signal', 'rsi_signal', 'donchian_signal',
    'compute_sortino', 'compute_sharpe', 'compute_cagr',
    'compute_max_drawdown', 'compute_calmar', 'compute_annual_volatility',
    'compute_drawdown_series',
]
missing = [f for f in required if not hasattr(module, f)]
if missing:
    raise ImportError(f'module.py missing: {missing}')
print('All module functions present.')

## 3. ETF Universe and Benchmark

In [ ]:
SECTOR_ETFS = {
    'XLF': 'Financials',
    'XLK': 'Technology',
    'XLV': 'Healthcare',
    'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',
    'XLE': 'Energy',
    'XLI': 'Industrials',
    'XLB': 'Materials',
    'XLU': 'Utilities',
    'IYR': 'Real Estate',   # IYR has history back to 2000; XLRE only from 2015
}
TICKERS   = list(SECTOR_ETFS.keys())
BENCHMARK = '^GSPC'

print('Sector ETF universe:')
for tk, name in SECTOR_ETFS.items():
    print(f'  {tk:<6} {name}')
print(f'Benchmark: {BENCHMARK}')

## 4. Data Loading and Cleaning

In [ ]:
import os

data_dir = pathlib.Path('data')
data_dir.mkdir(exist_ok=True)

try:
    import yfinance as yf
    _yf_ok = True
except ImportError:
    _yf_ok = False


def load_basket(tickers_list, csv_name, start='2010-01-01', end='2025-12-31'):
    csv_path = data_dir / csv_name
    if csv_path.exists():
        df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        present = [t for t in tickers_list if t in df.columns]
        df = df[present]
        print(f'  {csv_name}: loaded from cache ({df.shape})')
    else:
        try:
            df, _ = module.download_stock_price_data(tickers_list, start, end)
            df.to_csv(csv_path)
            print(f'  {csv_name}: downloaded via yahooquery ({df.shape})')
        except Exception as exc:
            if not _yf_ok:
                raise RuntimeError('pip install yfinance') from exc
            raw = yf.download(tickers_list, start=start, end=end,
                              progress=False, auto_adjust=True)
            df = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw
            if isinstance(df, pd.Series):
                df = df.to_frame(name=tickers_list[0])
            bidx = pd.date_range(start=start, end=end, freq='B')
            df = df.reindex(bidx).ffill().dropna(how='all')
            df.to_csv(csv_path)
            print(f'  {csv_name}: downloaded via yfinance ({df.shape})')
    df.index = pd.to_datetime(df.index)
    return df


print('Loading IS/OOS1 data (2010-2025)...')
df_all = load_basket(TICKERS, 'sector_etfs.csv', '2010-01-01', '2025-12-31')

print('Loading benchmark (2010-2025)...')
df_spx = load_basket([BENCHMARK], 'spx.csv', '2010-01-01', '2025-12-31')

print()
print(f'ETF frame : {df_all.shape}   ({df_all.index[0].date()} to {df_all.index[-1].date()})')
print(f'ETFs OK   : {list(df_all.columns)}')
missing_etfs = [t for t in TICKERS if t not in df_all.columns]
if missing_etfs:
    print(f'WARNING   : no data for {missing_etfs}')

## 5. In-Sample and Out-of-Sample Periods

In [ ]:
IS_START   = '2010-01-01'
IS_END     = '2019-12-31'
OOS1_START = '2020-01-01'
OOS1_END   = '2025-12-31'
OOS2_START = '2000-01-01'
OOS2_END   = '2009-12-31'


def split_df(df, start, end):
    return df[(df.index >= start) & (df.index <= end)].copy()


df_is   = split_df(df_all,  IS_START,   IS_END)
df_oos1 = split_df(df_all,  OOS1_START, OOS1_END)
df_spx_is   = split_df(df_spx, IS_START,   IS_END)
df_spx_oos1 = split_df(df_spx, OOS1_START, OOS1_END)

print(f'IS   : {IS_START} to {IS_END}   ({len(df_is)} trading days)')
print(f'OOS1 : {OOS1_START} to {OOS1_END}  ({len(df_oos1)} trading days)')
print(f'OOS2 : {OOS2_START} to {OOS2_END}  (extended data loaded in Section 13)')


def _spx_pv(df_basket, df_spx_ref):
    """S&P 500 normalised to 1.0 over the same date range as df_basket."""
    col = BENCHMARK if BENCHMARK in df_spx_ref.columns else df_spx_ref.columns[0]
    aligned = df_spx_ref[col].reindex(df_basket.index, method='ffill').to_numpy(dtype=float)
    return aligned / aligned[0]

## 6. Methodology

### Walk-forward validation
Parameters are selected exclusively on IS 2010–2019. OOS1 and OOS2 are held out completely — they are never inspected during optimisation.

### One-day execution lag
Signal computed at close of day *t* → position is active from day *t+1*.
Implementation:
```
position[t] = signal[t-1]
strategy_return[t] = position[t] × price_return[t]
```
This prevents look-ahead bias: today's signal earns tomorrow's return, not today's.

### What is a basis point (bps)?
> 1 bps = 0.01% = 0.0001. Transaction costs are expressed in bps because they are tiny fractions of the trade value. We use **10 bps per trade (one-way)**, which approximates broker commissions plus bid-ask spread for liquid ETFs. A round-trip (entry + exit) therefore costs 20 bps = 0.20%. Without a cost model a high-frequency strategy can look profitable in backtest but lose money in practice due to execution friction.

### Why are signals long-only?
> Signals output **0** (hold cash) or **1** (hold ETF). They never output **-1** (short). Shorting an ETF requires a margin account and a borrow fee, both outside the scope of this study. Long-only also avoids unlimited downside risk from short positions. The trade-off: the strategy sits in cash during sustained downtrends rather than profiting from them.

### Optimisation objective
Sortino ratio — penalises only downside deviation (Sortino & van der Meer, 1991).

## 7. Signal Definitions

All signals are implemented in `module.py` and return a DataFrame with:
- `signal`: 0 = cash, 1 = long (never -1; long-only)
- `position_change`: +1 = entry, -1 = exit, 0 = hold

### 7.1 Moving Average Crossover
Buy when short MA > long MA (Golden Cross); sell when long MA > short MA (Death Cross). Classic trend-following. Brock, Lakonishok & LeBaron (1992).

$$\text{Signal}_t = \mathbf{1}[MA_{\text{short}}(t) > MA_{\text{long}}(t)]$$

### 7.2 RSI
Buy when RSI drops below the oversold threshold; sell when it exceeds the overbought threshold. Mean-reversion logic. Wilder (1978).

$$RSI_t = 100 - \frac{100}{1 + \frac{\overline{\text{gain}}_t}{\overline{\text{loss}}_t}}$$

### 7.3 Donchian Channel Breakout
Buy when today's close exceeds the *previous* N-day high; sell when it falls below the *previous* N-day low. The `.shift(1)` on the rolling window avoids look-ahead bias inside the channel calculation. Turtle Trader canonical window: N = 55. Faith (2003).

$$\text{Signal}_t = \mathbf{1}[P_t > \max(P_{t-N}, \ldots, P_{t-1})]$$

In [ ]:
# Demo: plot each signal with default params on XLF IS data
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle('Signal Definitions — Default Parameters on XLF IS 2010–2019',
             fontsize=12, fontweight='bold')

_demo_etf = 'XLF' if 'XLF' in df_is.columns else df_is.columns[0]
_px_demo  = df_is[_demo_etf]

_demos = [
    (module.ma_signal,       {'short_window': 50, 'long_window': 200}, '7.1 MA (50/200)'),
    (module.rsi_signal,      {'period': 14, 'oversold': 30, 'overbought': 70}, '7.2 RSI (14/30/70)'),
    (module.donchian_signal, {'window': 55}, '7.3 Donchian (55)'),
]

for row, (fn, params, title) in enumerate(_demos):
    sig  = fn(_px_demo, **params)
    ax_p, ax_s = axes[row, 0], axes[row, 1]

    ax_p.plot(_px_demo.index, _px_demo.values, lw=1.0, color='black', label=_demo_etf)
    ax_p.fill_between(_px_demo.index,
                       _px_demo.min() * 0.97, _px_demo.max() * 1.03,
                       where=sig['signal'].values == 1,
                       alpha=0.15, color='green', label='Long')
    if 'short_ma' in sig.columns:
        ax_p.plot(sig.index, sig['short_ma'], lw=0.9, color='steelblue',
                  label='MA short', alpha=0.8)
        ax_p.plot(sig.index, sig['long_ma'],  lw=0.9, color='tomato',
                  label='MA long',  alpha=0.8)
    if 'entry_high' in sig.columns:
        ax_p.plot(sig.index, sig['entry_high'], '--', lw=0.9, color='green', alpha=0.7)
        ax_p.plot(sig.index, sig['exit_low'],   '--', lw=0.9, color='red',   alpha=0.7)
    ax_p.set_title(title, fontsize=10, fontweight='bold')
    ax_p.legend(fontsize=7); ax_p.grid(True, alpha=0.3)

    if 'rsi' in sig.columns:
        ax_s.plot(sig.index, sig['rsi'], color='purple', lw=1.0)
        ax_s.axhline(30, color='green', lw=0.8, linestyle='--', label='Oversold 30')
        ax_s.axhline(70, color='red',   lw=0.8, linestyle='--', label='Overbought 70')
        ax_s.set_ylim(0, 100); ax_s.set_ylabel('RSI')
    else:
        ax_s.step(sig.index, sig['signal'], color='navy', lw=1.2, where='post')
        ax_s.set_yticks([0, 1]); ax_s.set_ylabel('Signal (0=cash, 1=long)')
    n_trades   = int(np.sum(sig['position_change'] > 0))
    active_pct = sig['signal'].mean() * 100
    ax_s.set_title(f'Entries={n_trades}   Active={active_pct:.1f}%', fontsize=9)
    ax_s.legend(fontsize=7); ax_s.grid(True, alpha=0.3)
    ax_s.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax_s.xaxis.set_major_locator(mdates.YearLocator(3))

plt.tight_layout(); plt.show()

## 8. Parameter Grids

In [ ]:
# MA Crossover: sweep short and long windows (only pairs where short < long)
ma_grid = [
    {'short_window': short, 'long_window': long}
    for short in [20, 50, 75]
    for long  in [100, 150, 200, 250]
    if short < long
]

# RSI: period fixed at 14 (Wilder standard); sweep entry/exit thresholds
rsi_grid = [
    {'period': 14, 'oversold': os_, 'overbought': ob}
    for os_ in [20, 25, 30, 35, 40]
    for ob  in [60, 65, 70, 75, 80]
    if os_ < ob
]

# Donchian Channel: sweep channel width
donchian_grid = [
    {'window': w}
    for w in [20, 40, 55, 75, 100, 150, 200]
]

SIGNAL_GRIDS = [
    ('MA Crossover', module.ma_signal,       ma_grid),
    ('RSI',          module.rsi_signal,      rsi_grid),
    ('Donchian',     module.donchian_signal, donchian_grid),
]

print(f'MA Crossover grid : {len(ma_grid)} parameter combinations')
print(f'RSI grid          : {len(rsi_grid)} parameter combinations')
print(f'Donchian grid     : {len(donchian_grid)} parameter combinations')
print(f'Total IS backtests: {(len(ma_grid) + len(rsi_grid) + len(donchian_grid)) * len(TICKERS)}')

## 9. Backtest Engine with One-Day Signal Lag

In [ ]:
TRADE_COST = 0.001   # 10 bps one-way


def single_etf_portfolio_value(signal_fn, price_series, params,
                                trade_cost=TRADE_COST):
    """Lagged backtest for a single ETF price series.

    One-day lag: position[t] = signal[t-1] so today's signal earns tomorrow's return.
    Transaction cost charged each time the position changes.

    Returns
    -------
    gross_pv, net_pv, daily_net, position, trade
    """
    px = price_series.to_numpy(dtype=float)
    daily_ret = np.concatenate(([0.0], px[1:] / px[:-1] - 1))

    sig        = signal_fn(price_series, **params)
    signal_arr = sig['signal'].to_numpy(dtype=float)

    # One-day execution lag
    position = np.concatenate(([0.0], signal_arr[:-1]))

    previous_position = np.concatenate(([0.0], position[:-1]))
    trade = np.abs(position - previous_position)

    daily_gross = position  * daily_ret
    daily_cost  = trade     * trade_cost
    daily_net   = daily_gross - daily_cost

    gross_pv = np.cumprod(1.0 + daily_gross)
    net_pv   = np.cumprod(1.0 + daily_net)

    return gross_pv, net_pv, daily_net, position, trade


print(f'TRADE_COST = {TRADE_COST*10000:.0f} bps one-way  ({TRADE_COST*2*100:.2f}% round-trip)')

## 10. Performance Metrics

In [ ]:
def performance_metrics_from_pv(pv):
    """Return dict of annualised performance metrics from a portfolio-value series."""
    dr = np.concatenate(([0.0], pv[1:] / pv[:-1] - 1))
    return {
        'Net Return': float(pv[-1] / pv[0] - 1),
        'CAGR':       float(module.compute_cagr(pv)),
        'Ann. Vol':   float(module.compute_annual_volatility(dr[1:])),
        'Sharpe':     float(module.compute_sharpe(dr[1:])),
        'Sortino':    float(module.compute_sortino(dr[1:])),
        'Calmar':     float(module.compute_calmar(pv)),
        'MaxDD':      float(module.compute_max_drawdown(pv)),
    }


def full_metrics(pv_net, pv_spx, label):
    """Print strategy vs S&P 500 metrics side-by-side."""
    m_s = performance_metrics_from_pv(pv_net)
    m_b = performance_metrics_from_pv(pv_spx)
    W = 54
    print(f'  {"="*W}')
    print(f'  {label}')
    print(f'  {"-"*W}')
    print(f'  {"Metric":<22} {"Strategy":>12}  {"S&P 500":>10}')
    print(f'  {"-"*W}')
    for name, (sv, bv) in [
        ('Net Return',  (m_s['Net Return'], m_b['Net Return'])),
        ('CAGR',        (m_s['CAGR'],       m_b['CAGR'])),
        ('Ann. Vol',    (m_s['Ann. Vol'],    m_b['Ann. Vol'])),
        ('Sharpe',      (m_s['Sharpe'],      m_b['Sharpe'])),
        ('Sortino',     (m_s['Sortino'],     m_b['Sortino'])),
        ('Calmar',      (m_s['Calmar'],      m_b['Calmar'])),
        ('Max Drawdown',(m_s['MaxDD'],       m_b['MaxDD'])),
    ]:
        pct = any(x in name for x in ['Return', 'Vol', 'Drawdown', 'CAGR'])
        fmt = '.2%' if pct else '.3f'
        sv_str = format(sv, fmt)
        bv_str = format(bv, fmt)
        print(f'  {name:<22} {sv_str:>12}  {bv_str:>10}')
    print(f'  {"-"*W}')
    print()


print('Performance helpers loaded.')

## 11. In-Sample Optimisation: 2010–2019

In [ ]:
def optimise_signal_on_etf(signal_fn, price_is, param_grid):
    """Return (best_params, best_sortino) by maximising IS net Sortino ratio.
    Parameters are never changed after this step.
    """
    best_sortino = -np.inf
    best_params  = None
    for params in param_grid:
        try:
            _, _, daily_net, _, _ = single_etf_portfolio_value(
                signal_fn, price_is, params)
            s = module.compute_sortino(daily_net[1:])
            if s == s and s > best_sortino:
                best_sortino = s
                best_params  = dict(params)
        except Exception:
            continue
    return best_params, float(best_sortino) if best_sortino > -np.inf else np.nan


# Run IS optimisation for every (signal, ETF) pair
# is_opt[signal_name][ticker] = {'params': ..., 'sortino': ..., 'pv': ...}
is_opt = {sn: {} for sn, _, _ in SIGNAL_GRIDS}

for sig_name, sig_fn, param_grid in SIGNAL_GRIDS:
    print(f'  Optimising {sig_name} ({len(param_grid)} params × {len(TICKERS)} ETFs)...')
    for tk in TICKERS:
        if tk not in df_is.columns:
            is_opt[sig_name][tk] = {'params': None, 'sortino': np.nan, 'pv': None}
            continue
        best_p, best_s = optimise_signal_on_etf(sig_fn, df_is[tk], param_grid)
        if best_p is not None:
            _, pv, _, _, _ = single_etf_portfolio_value(sig_fn, df_is[tk], best_p)
        else:
            pv = None
        is_opt[sig_name][tk] = {'params': best_p, 'sortino': best_s, 'pv': pv}

print('IS optimisation complete.')

In [ ]:
# IS Sortino results table
print('IS Sortino Ratios — optimal params per ETF per signal (2010-2019)')
print(f'  {"ETF":<6} {"Sector":<24}', end='')
for sn, _, _ in SIGNAL_GRIDS:
    print(f' {sn:>16}', end='')
print()
print('  ' + '-' * 74)
for tk in TICKERS:
    sector = SECTOR_ETFS.get(tk, tk)
    print(f'  {tk:<6} {sector:<24}', end='')
    for sn, _, _ in SIGNAL_GRIDS:
        s = is_opt[sn][tk]['sortino']
        print(f' {s:>16.3f}' if s == s else f' {"NaN":>16}', end='')
    print()
print('  ' + '-' * 74)
print()
for sn, _, _ in SIGNAL_GRIDS:
    ranks = [(tk, is_opt[sn][tk]['sortino']) for tk in TICKERS
             if is_opt[sn][tk]['sortino'] == is_opt[sn][tk]['sortino']]
    ranks.sort(key=lambda x: x[1], reverse=True)
    best_tk, best_s = ranks[0]
    best_p = is_opt[sn][best_tk]['params']
    print(f'  IS best for {sn:<16}: {best_tk} ({SECTOR_ETFS[best_tk]})'
          f'  Sortino={best_s:.3f}  params={best_p}')

In [ ]:
# IS Sortino heatmap (3 signals x 10 ETFs)
_mat_is = np.full((len(SIGNAL_GRIDS), len(TICKERS)), np.nan)
for r, (sn, _, _) in enumerate(SIGNAL_GRIDS):
    for c, tk in enumerate(TICKERS):
        _mat_is[r, c] = is_opt[sn][tk]['sortino']

fig, ax = plt.subplots(figsize=(13, 4))
vmin, vmax = float(np.nanmin(_mat_is)), float(np.nanmax(_mat_is))
im = ax.imshow(_mat_is, aspect='auto', cmap='RdYlGn', vmin=vmin, vmax=vmax)
ax.set_xticks(range(len(TICKERS)))
ax.set_xticklabels([f'{tk}\n{SECTOR_ETFS[tk]}' for tk in TICKERS], fontsize=8)
ax.set_yticks(range(len(SIGNAL_GRIDS)))
ax.set_yticklabels([sn for sn, _, _ in SIGNAL_GRIDS], fontsize=9)
mid = (vmin + vmax) / 2
for r in range(_mat_is.shape[0]):
    for c in range(_mat_is.shape[1]):
        v = _mat_is[r, c]
        if v == v:
            tc = 'black' if v > mid else 'white'
            ax.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=8, color=tc)
plt.colorbar(im, ax=ax, label='IS Sortino (optimal params per cell)')
ax.set_title('IS Sortino Heatmap (2010-2019) — Optimal Params per Signal x ETF',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

## 12. Out-of-Sample Validation 1: 2020–2025

IS-optimal parameters applied unchanged. Period covers COVID-19 (Feb–Mar 2020), 2022 rate-hike drawdown, and 2023–2025 AI bull run.

In [ ]:
# Apply IS-optimal params on OOS1 for each (signal, ETF) pair
oos1_perf = {sn: {} for sn, _, _ in SIGNAL_GRIDS}

for sig_name, sig_fn, _ in SIGNAL_GRIDS:
    for tk in TICKERS:
        best_p = is_opt[sig_name][tk]['params']
        if best_p is None or tk not in df_oos1.columns:
            oos1_perf[sig_name][tk] = {'sortino': np.nan, 'pv': None}
            continue
        try:
            _, pv, daily_net, _, _ = single_etf_portfolio_value(
                sig_fn, df_oos1[tk], best_p)
            m = performance_metrics_from_pv(pv)
            m['pv'] = pv
            oos1_perf[sig_name][tk] = m
        except Exception:
            oos1_perf[sig_name][tk] = {'sortino': np.nan, 'pv': None}

print('OOS1 Sortino Ratios (2020-2025) — IS-optimal params frozen:')
print(f'  {"ETF":<6} {"Sector":<24}', end='')
for sn, _, _ in SIGNAL_GRIDS:
    print(f' {sn:>16}', end='')
print()
print('  ' + '-' * 74)
for tk in TICKERS:
    print(f'  {tk:<6} {SECTOR_ETFS.get(tk,tk):<24}', end='')
    for sn, _, _ in SIGNAL_GRIDS:
        s = oos1_perf[sn][tk].get('Sortino', np.nan)
        print(f' {s:>16.3f}' if s == s else f' {"NaN":>16}', end='')
    print()

## 13. Out-of-Sample Validation 2: 2000–2009

Pre-sample stress-test spanning dot-com bust (2000–2002), mid-decade recovery, and early GFC (2007–2009). Extended data loaded separately.

In [ ]:
print('Loading extended data (2000-2025) for all 10 ETFs...')
df_ext = load_basket(TICKERS, 'sector_etfs_ext.csv', '2000-01-01', '2025-12-31')

print('Loading extended S&P 500...')
df_spx_ext = load_basket([BENCHMARK], 'spx_ext.csv', '2000-01-01', '2025-12-31')

df_oos2     = split_df(df_ext,     OOS2_START, OOS2_END)
df_spx_oos2 = split_df(df_spx_ext, OOS2_START, OOS2_END)

print()
print(f'OOS2 frame : {df_oos2.shape}  ({df_oos2.index[0].date()} to {df_oos2.index[-1].date()})')
missing_oos2 = [t for t in TICKERS if t not in df_oos2.columns or df_oos2[t].isna().all()]
if missing_oos2:
    print(f'WARNING: no OOS2 data for {missing_oos2}')

In [ ]:
# Apply IS-optimal params on OOS2 for each (signal, ETF) pair
oos2_perf = {sn: {} for sn, _, _ in SIGNAL_GRIDS}

for sig_name, sig_fn, _ in SIGNAL_GRIDS:
    for tk in TICKERS:
        best_p = is_opt[sig_name][tk]['params']
        if best_p is None or tk not in df_oos2.columns:
            oos2_perf[sig_name][tk] = {'sortino': np.nan, 'pv': None}
            continue
        series_oos2 = df_oos2[tk].dropna()
        if len(series_oos2) < 100:
            oos2_perf[sig_name][tk] = {'sortino': np.nan, 'pv': None}
            continue
        try:
            _, pv, daily_net, _, _ = single_etf_portfolio_value(
                sig_fn, series_oos2, best_p)
            m = performance_metrics_from_pv(pv)
            m['pv'] = pv
            m['idx'] = series_oos2.index
            oos2_perf[sig_name][tk] = m
        except Exception:
            oos2_perf[sig_name][tk] = {'sortino': np.nan, 'pv': None}

print('OOS2 Sortino Ratios (2000-2009) — IS-optimal params frozen:')
print(f'  {"ETF":<6} {"Sector":<24}', end='')
for sn, _, _ in SIGNAL_GRIDS:
    print(f' {sn:>16}', end='')
print()
print('  ' + '-' * 74)
for tk in TICKERS:
    print(f'  {tk:<6} {SECTOR_ETFS.get(tk,tk):<24}', end='')
    for sn, _, _ in SIGNAL_GRIDS:
        s = oos2_perf[sn][tk].get('Sortino', np.nan)
        print(f' {s:>16.3f}' if s == s else f' {"NaN":>16}', end='')
    print()

## 14. Combined IS / OOS Results Table

In [ ]:
rows = []
for sig_name, sig_fn, _ in SIGNAL_GRIDS:
    for tk in TICKERS:
        bp  = is_opt[sig_name][tk]['params']
        is_s   = is_opt[sig_name][tk]['sortino']
        oos1_s = oos1_perf[sig_name][tk].get('Sortino',   np.nan)
        oos2_s = oos2_perf[sig_name][tk].get('Sortino',   np.nan)
        oos1_sh = oos1_perf[sig_name][tk].get('Sharpe',   np.nan)
        oos2_sh = oos2_perf[sig_name][tk].get('Sharpe',   np.nan)
        is_sh   = np.nan  # IS Sharpe from pv
        if is_opt[sig_name][tk]['pv'] is not None:
            is_sh = performance_metrics_from_pv(is_opt[sig_name][tk]['pv'])['Sharpe']
        rows.append({
            'Signal':          sig_name,
            'ETF':             tk,
            'Sector':          SECTOR_ETFS.get(tk, tk),
            'Best Params':     str(bp),
            'IS Sortino':      is_s,
            'OOS1 Sortino':    oos1_s,
            'OOS2 Sortino':    oos2_s,
            'IS Sharpe':       is_sh,
            'OOS1 Sharpe':     oos1_sh,
            'OOS2 Sharpe':     oos2_sh,
            'IS CAGR':         performance_metrics_from_pv(is_opt[sig_name][tk]['pv'])['CAGR']
                               if is_opt[sig_name][tk]['pv'] is not None else np.nan,
            'OOS1 CAGR':       oos1_perf[sig_name][tk].get('CAGR',   np.nan),
            'OOS2 CAGR':       oos2_perf[sig_name][tk].get('CAGR',   np.nan),
            'IS MaxDD':        performance_metrics_from_pv(is_opt[sig_name][tk]['pv'])['MaxDD']
                               if is_opt[sig_name][tk]['pv'] is not None else np.nan,
            'OOS1 MaxDD':      oos1_perf[sig_name][tk].get('MaxDD',  np.nan),
            'OOS2 MaxDD':      oos2_perf[sig_name][tk].get('MaxDD',  np.nan),
        })

df_results = pd.DataFrame(rows)
df_results['Min OOS Sortino'] = df_results[['OOS1 Sortino', 'OOS2 Sortino']].min(axis=1)
df_results['Avg OOS Sortino'] = df_results[['OOS1 Sortino', 'OOS2 Sortino']].mean(axis=1)

# Display sorted by signal then Min OOS Sortino
for sig_name, _, _ in SIGNAL_GRIDS:
    sub = df_results[df_results['Signal'] == sig_name].sort_values(
        'Min OOS Sortino', ascending=False)
    print(f'\n  {sig_name}')
    print(sub[['ETF','Sector','IS Sortino','OOS1 Sortino','OOS2 Sortino',
               'Min OOS Sortino','IS CAGR','OOS1 CAGR','OOS2 CAGR']].to_string(index=False))

## 15. Heatmaps and Visual Comparison

In [ ]:
def _make_heatmap(ax, mat, col_labels, row_labels, title, star_mask=None):
    vmin = float(np.nanmin(mat)) if not np.all(np.isnan(mat)) else -1
    vmax = float(np.nanmax(mat)) if not np.all(np.isnan(mat)) else  1
    im = ax.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, fontsize=7, rotation=30, ha='right')
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=9)
    ax.set_title(title, fontsize=9, fontweight='bold')
    mid = (vmin + vmax) / 2
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            v = mat[r, c]
            if v == v:
                tc  = 'black' if v > mid else 'white'
                mrk = ' *' if (star_mask is not None and star_mask[r, c]) else ''
                ax.text(c, r, f'{v:.2f}{mrk}', ha='center', va='center',
                        fontsize=7, color=tc)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


sig_names = [sn for sn, _, _ in SIGNAL_GRIDS]
n_sig = len(sig_names)
n_etf = len(TICKERS)

mat_is   = np.full((n_sig, n_etf), np.nan)
mat_oos1 = np.full((n_sig, n_etf), np.nan)
mat_oos2 = np.full((n_sig, n_etf), np.nan)

for r, sn in enumerate(sig_names):
    for c, tk in enumerate(TICKERS):
        mat_is[r,c]   = is_opt[sn][tk]['sortino']
        mat_oos1[r,c] = oos1_perf[sn][tk].get('Sortino', np.nan)
        mat_oos2[r,c] = oos2_perf[sn][tk].get('Sortino', np.nan)

mat_min_oos = np.fmin(mat_oos1, mat_oos2)   # element-wise NaN-safe min

# Mark best ETF per signal on IS heatmap
star_is = np.zeros((n_sig, n_etf), dtype=bool)
for r in range(n_sig):
    if not np.all(np.isnan(mat_is[r])):
        star_is[r, int(np.nanargmax(mat_is[r]))] = True

etf_labels = [f'{tk}\n{SECTOR_ETFS[tk]}' for tk in TICKERS]

fig, axes = plt.subplots(1, 4, figsize=(22, 3.5))
_make_heatmap(axes[0], mat_is,      etf_labels, sig_names,
              'IS Sortino (2010-2019)\n* = IS best per signal', star_mask=star_is)
_make_heatmap(axes[1], mat_oos1,    etf_labels, sig_names, 'OOS1 Sortino (2020-2025)')
_make_heatmap(axes[2], mat_oos2,    etf_labels, sig_names, 'OOS2 Sortino (2000-2009)')
_make_heatmap(axes[3], mat_min_oos, etf_labels, sig_names, 'Min OOS Sortino\n(min of OOS1, OOS2)')

fig.suptitle('Signal x ETF Performance Heatmaps — All Periods',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 16. Strategy Selection

**Selection rule:** For each signal, the ETF with the **highest IS Sortino** is the primary candidate. Selection is validated by checking that both OOS periods show positive Sortino. Robustness is then ranked by minimum OOS Sortino.

> OOS1 and OOS2 are never used to choose parameters — only to validate whether the IS-selected strategy generalises.

In [ ]:
selected = {}  # selected[sig_name] = {'etf', 'params', 'is_sortino', 'oos1_sortino', 'oos2_sortino'}

W = 90
print('=' * W)
print('  STRATEGY SELECTION SCORECARD')
print('=' * W)
print(f'  {"Signal":<16} {"ETF":<6} {"IS Sort":>9} {"O1 Sort":>9} {"O2 Sort":>9}'
      f' {"MinOOS":>8} {"O1>0":>6} {"O2>0":>6}  Decision')
print('  ' + '-' * (W - 2))

for sig_name, _, _ in SIGNAL_GRIDS:
    sub = df_results[df_results['Signal'] == sig_name].copy()
    sub = sub.sort_values('IS Sortino', ascending=False)

    chosen_etf    = None
    chosen_params = None
    for _, row in sub.iterrows():
        tk   = row['ETF']
        s_is = row['IS Sortino']
        s_o1 = row['OOS1 Sortino']
        s_o2 = row['OOS2 Sortino']
        s_mn = row['Min OOS Sortino']
        o1ok = 'Y' if s_o1 > 0 else 'N'
        o2ok = 'Y' if s_o2 > 0 else 'N'
        bp   = is_opt[sig_name][tk]['params']
        if chosen_etf is None and s_o1 > 0 and s_o2 > 0:
            chosen_etf    = tk
            chosen_params = bp
            tag = '  <-- SELECTED'
        else:
            tag = ''
        print(f'  {sig_name:<16} {tk:<6} {s_is:>9.3f} {s_o1:>9.3f} {s_o2:>9.3f}'
              f' {s_mn:>8.3f} {o1ok:>6} {o2ok:>6}  {tag}')

    if chosen_etf is None:
        top_row = sub.iloc[0]
        chosen_etf    = top_row['ETF']
        chosen_params = is_opt[sig_name][chosen_etf]['params']
        print(f'  -> No ETF passed both OOS filters; defaulting to IS best: {chosen_etf}')
    selected[sig_name] = {
        'etf': chosen_etf, 'params': chosen_params,
        'sig_fn': next(fn for sn, fn, _ in SIGNAL_GRIDS if sn == sig_name),
        'is_sortino':   is_opt[sig_name][chosen_etf]['sortino'],
        'oos1_sortino': oos1_perf[sig_name][chosen_etf].get('Sortino', np.nan),
        'oos2_sortino': oos2_perf[sig_name][chosen_etf].get('Sortino', np.nan),
    }

print('=' * W)
print()
print('Selected strategies:')
for sig_name, d in selected.items():
    print(f'  {sig_name:<16}: {d["etf"]} ({SECTOR_ETFS[d["etf"]]})  params={d["params"]}')

## 17. Benchmark Comparison Against S&P 500

In [ ]:
# Full 2010-2025 period using IS-optimal params
print('Full-period benchmark comparison (2010-2025, net of 10 bps):')
print()

for sig_name, d in selected.items():
    tk      = d['etf']
    sig_fn  = d['sig_fn']
    params  = d['params']
    series  = df_all[tk] if tk in df_all.columns else None
    if series is None or params is None:
        print(f'  {sig_name}: no data / no params')
        continue
    _, pv_net, _, _, _ = single_etf_portfolio_value(sig_fn, series, params)
    pv_spx = _spx_pv(df_all[[tk]], df_spx)
    d['pv_full'] = pv_net
    d['idx_full'] = df_all.index
    full_metrics(pv_net, pv_spx,
                 f'{sig_name} | {tk} ({SECTOR_ETFS[tk]}) | 2010-2025 net 10 bps')

## 18. Equity Curve Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Cumulative Net Portfolio Value vs S&P 500 (Log Scale, 2010–2025)\n'
             'Grey vertical line = OOS1 start (2020). Shaded area = OOS1.',
             fontsize=11, fontweight='bold')

_colors = ['navy', 'forestgreen', 'darkorchid']

for ax, (sig_name, d), col in zip(axes, selected.items(), _colors):
    tk      = d['etf']
    sig_fn  = d['sig_fn']
    params  = d['params']
    if params is None or tk not in df_is.columns:
        ax.set_title(f'{sig_name} — no data'); continue

    # IS
    _, pv_is, _, _, _ = single_etf_portfolio_value(sig_fn, df_is[tk],   params)
    _, pv_o1, _, _, _ = single_etf_portfolio_value(sig_fn, df_oos1[tk], params)

    idx_is = df_is.index.to_numpy()
    idx_o1 = df_oos1.index.to_numpy()
    idx_all = np.concatenate([idx_is, idx_o1])
    pv_all  = np.concatenate([pv_is, pv_o1 / pv_o1[0] * pv_is[-1]])

    spx_is_arr = _spx_pv(df_is[[tk]],   df_spx_is)
    spx_o1_arr = _spx_pv(df_oos1[[tk]], df_spx_oos1)
    spx_all    = np.concatenate([spx_is_arr, spx_o1_arr / spx_o1_arr[0] * spx_is_arr[-1]])

    ax.semilogy(idx_all, pv_all,  color=col,     lw=1.8, label='Strategy (Net)')
    ax.semilogy(idx_all, spx_all, color='tomato', lw=1.4, linestyle='--',
                label='S&P 500', alpha=0.85)
    ax.axvline(pd.Timestamp(OOS1_START), color='grey', lw=1.2, linestyle=':', alpha=0.8)
    ax.axvspan(pd.Timestamp(OOS1_START), idx_o1[-1], alpha=0.06, color='grey')
    ax.set_title(f'{sig_name}\n{tk} ({SECTOR_ETFS[tk]})', fontsize=10, fontweight='bold')
    ax.set_ylabel('Portfolio Value (log)', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3))
    ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.2, linestyle='--')

plt.tight_layout(); plt.show()

## 19. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
fig.suptitle('Drawdown Profiles — Net Strategy vs S&P 500 (2010–2025)',
             fontsize=12, fontweight='bold')

_covid_s = pd.Timestamp('2020-02-19'); _covid_e = pd.Timestamp('2020-03-23')
_hike_s  = pd.Timestamp('2022-01-03'); _hike_e  = pd.Timestamp('2022-12-30')

for ax, (sig_name, d), col in zip(axes, selected.items(), _colors):
    tk      = d['etf']
    sig_fn  = d['sig_fn']
    params  = d['params']
    if params is None or tk not in df_is.columns:
        ax.set_title(f'{sig_name} — no data'); continue

    _, pv_is, _, _, _ = single_etf_portfolio_value(sig_fn, df_is[tk],   params)
    _, pv_o1, _, _, _ = single_etf_portfolio_value(sig_fn, df_oos1[tk], params)
    idx_all = np.concatenate([df_is.index.to_numpy(), df_oos1.index.to_numpy()])
    pv_all  = np.concatenate([pv_is, pv_o1 / pv_o1[0] * pv_is[-1]])

    spx_is_arr = _spx_pv(df_is[[tk]],   df_spx_is)
    spx_o1_arr = _spx_pv(df_oos1[[tk]], df_spx_oos1)
    spx_all    = np.concatenate([spx_is_arr, spx_o1_arr / spx_o1_arr[0] * spx_is_arr[-1]])

    dd_s = module.compute_drawdown_series(pv_all)  * 100
    dd_b = module.compute_drawdown_series(spx_all) * 100

    ax.fill_between(idx_all, dd_s, 0, color=col,     alpha=0.30, label='Strategy DD')
    ax.fill_between(idx_all, dd_b, 0, color='tomato', alpha=0.18, label='S&P 500 DD')
    ax.plot(idx_all, dd_s, color=col,     lw=0.9, alpha=0.8)
    ax.plot(idx_all, dd_b, color='tomato', lw=0.9, alpha=0.7, linestyle='--')
    ax.axvspan(_covid_s, _covid_e, alpha=0.15, color='red',    label='COVID crash')
    ax.axvspan(_hike_s,  _hike_e,  alpha=0.10, color='orange', label='Rate-hike cycle')
    ax.axvline(pd.Timestamp(OOS1_START), color='grey', lw=1.1, linestyle=':', alpha=0.8)
    ax.set_title(f'{sig_name} | {tk} ({SECTOR_ETFS[tk]})', fontsize=10, fontweight='bold')
    ax.set_ylabel('Drawdown (%)', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.legend(fontsize=7, loc='lower left', ncol=2)
    ax.grid(True, alpha=0.25, linestyle='--')

plt.tight_layout(); plt.show()

## 20. Robustness Checks

1. **Neighbourhood test** — OOS1 Sortino in a 3×3 neighbourhood around IS optimum
2. **Cost sensitivity** — net Sortino at 0 / 5 / 10 / 20 bps one-way
3. **Rolling 252-day Sharpe** — time-varying performance monitoring
4. **Annual return decomposition** — year-by-year strategy return

In [ ]:
# -- 20.1 Neighbourhood Test --

def neighbourhood_test(sig_fn, df_oos, param_grid_2d, p1, p2, opt_p1, opt_p2, label):
    p1v = param_grid_2d[p1]; p2v = param_grid_2d[p2]
    i1  = p1v.index(opt_p1) if opt_p1 in p1v else -1
    i2  = p2v.index(opt_p2) if opt_p2 in p2v else -1
    nb1 = list(dict.fromkeys([p1v[max(0,i1-1)], opt_p1, p1v[min(len(p1v)-1,i1+1)]]))
    nb2 = list(dict.fromkeys([p2v[max(0,i2-1)], opt_p2, p2v[min(len(p2v)-1,i2+1)]]))
    print(f'  {label}')
    print(f'  OOS1 Sortino  |  {p1}={opt_p1}  {p2}={opt_p2}  (* = IS optimum)')
    header = f'  {p1:<14} ' + ''.join(f'{str(v):>10}' for v in nb2)
    sep    = '  ' + '-' * (len(header) - 2)
    print(sep); print(header); print(sep)
    for v1 in nb1:
        row = f'  {str(v1):<14}'
        for v2 in nb2:
            try:
                kw = {p1: v1, p2: v2}
                _, _, dn, _, _ = single_etf_portfolio_value(sig_fn, df_oos, kw)
                s   = module.compute_sortino(dn[1:])
                tag = '* ' if (v1 == opt_p1 and v2 == opt_p2) else '  '
                row += f'  {s:>6.3f}{tag}'
            except Exception:
                row += f'  {"ERR":>8}'
        print(row)
    print(sep); print()


_ma_data  = selected['MA Crossover']
_rsi_data = selected['RSI']
_don_data = selected['Donchian']

if _ma_data['params']:
    neighbourhood_test(
        module.ma_signal, df_oos1[_ma_data['etf']],
        {'short_window': [20, 50, 75], 'long_window': [100, 150, 200, 250]},
        'short_window', 'long_window',
        _ma_data['params']['short_window'], _ma_data['params']['long_window'],
        f'MA Crossover | {_ma_data["etf"]} OOS1 2020-2025')

if _rsi_data['params']:
    neighbourhood_test(
        module.rsi_signal, df_oos1[_rsi_data['etf']],
        {'oversold': [20, 25, 30, 35, 40], 'overbought': [60, 65, 70, 75, 80]},
        'oversold', 'overbought',
        _rsi_data['params']['oversold'], _rsi_data['params']['overbought'],
        f'RSI | {_rsi_data["etf"]} OOS1 2020-2025')

if _don_data['params']:
    opt_w = _don_data['params']['window']
    print(f'  Donchian | {_don_data["etf"]} OOS1 Sortino for all windows (IS opt={opt_w}*):')
    print(f'  {"window":<14} {"OOS1 Sortino":>12}')
    print('  ' + '-' * 30)
    for w in [20, 40, 55, 75, 100, 150, 200]:
        try:
            _, _, dn, _, _ = single_etf_portfolio_value(
                module.donchian_signal, df_oos1[_don_data['etf']], {'window': w})
            s = module.compute_sortino(dn[1:])
            tag = ' *' if w == opt_w else ''
            print(f'  {str(w):<14} {s:>12.3f}{tag}')
        except Exception:
            print(f'  {str(w):<14}       ERR')
    print('  ' + '-' * 30)

In [ ]:
# -- 20.2 Transaction Cost Sensitivity --
costs  = [0.0, 0.0005, 0.001, 0.002]
c_lbls = ['0 bps', '5 bps', '10 bps (base)', '20 bps']

W2 = 66
print('=' * W2)
print('  Cost Sensitivity | OOS1 Net Sortino')
print('=' * W2)
print(f'  {"Cost":<16}', end='')
for sig_name in selected:
    print(f' {sig_name:>16}', end='')
print()
print('  ' + '-' * (W2 - 2))

for tc, lbl in zip(costs, c_lbls):
    print(f'  {lbl:<16}', end='')
    for sig_name, d in selected.items():
        tk = d['etf']; fn = d['sig_fn']; p = d['params']
        if p is None or tk not in df_oos1.columns:
            print(f' {"NaN":>16}', end='')
        else:
            _, _, dn, _, _ = single_etf_portfolio_value(fn, df_oos1[tk], p, tc)
            s = module.compute_sortino(dn[1:])
            print(f' {s:>16.3f}', end='')
    print()
print('=' * W2)

In [ ]:
# -- 20.3 Rolling 252-Day Sharpe --
ROLL = 252

def rolling_sharpe_series(pv):
    n  = len(pv)
    dr = np.concatenate(([0.0], pv[1:] / pv[:-1] - 1))
    rs = np.full(n, np.nan)
    for i in range(ROLL, n):
        w  = dr[i - ROLL:i]
        mu = np.sum(w) / ROLL
        sg = np.sqrt(np.sum((w - mu) ** 2) / ROLL)
        if sg > 1e-10:
            rs[i] = mu / sg * np.sqrt(ROLL)
    return rs


fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
fig.suptitle('Rolling 252-Day Annualised Sharpe (Net, 2010–2025)',
             fontsize=12, fontweight='bold')

for ax, (sig_name, d), col in zip(axes, selected.items(), _colors):
    tk = d['etf']; fn = d['sig_fn']; p = d['params']
    if p is None or tk not in df_is.columns:
        ax.set_title(f'{sig_name} — no data'); continue

    _, pv_is, _, _, _ = single_etf_portfolio_value(fn, df_is[tk],   p)
    _, pv_o1, _, _, _ = single_etf_portfolio_value(fn, df_oos1[tk], p)
    idx_all = np.concatenate([df_is.index.to_numpy(), df_oos1.index.to_numpy()])
    pv_all  = np.concatenate([pv_is, pv_o1 / pv_o1[0] * pv_is[-1]])

    rs = rolling_sharpe_series(pv_all)
    ax.plot(idx_all, rs, color=col, lw=1.3, label=sig_name)
    ax.axhline(0, color='black', lw=0.8, alpha=0.6)
    ax.axhline(1, color='green', lw=0.7, linestyle='--', alpha=0.5, label='SR=1')
    ax.fill_between(idx_all, rs, 0, where=(rs > 0), alpha=0.12, color=col)
    ax.fill_between(idx_all, rs, 0, where=(rs < 0), alpha=0.18, color='red')
    ax.axvline(pd.Timestamp(OOS1_START), color='grey', lw=1.1, linestyle=':', alpha=0.8)
    ax.set_title(f'{sig_name} | {tk} ({SECTOR_ETFS[tk]})', fontsize=10, fontweight='bold')
    ax.set_ylabel('Rolling Sharpe', fontsize=9)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.grid(True, alpha=0.25, linestyle='--')

plt.tight_layout(); plt.show()

In [ ]:
# -- 20.4 Annual Return Decomposition --
for sig_name, d in selected.items():
    tk = d['etf']; fn = d['sig_fn']; p = d['params']
    if p is None or tk not in df_all.columns:
        continue
    series = df_all[tk]
    _, _, daily_net, pos, _ = single_etf_portfolio_value(fn, series, p)
    years   = pd.to_datetime(series.index).year.to_numpy()
    n_pos   = 0; n_tot = 0
    print(f'{"="*58}')
    print(f'  {sig_name} | {tk} ({SECTOR_ETFS[tk]}) | params={p}')
    print(f'{"="*58}')
    print(f'  {"Year":<6} {"Net Return":>12} {"Active Days":>13}  Period')
    print(f'  {"-"*50}')
    for yr in sorted(np.unique(years)):
        mask   = years == yr
        yr_ret = float(np.sum(daily_net[mask]))
        yr_act = int(np.sum(pos[mask] > 0))
        period = 'OOS1' if yr >= 2020 else 'IS  '
        n_pos += int(yr_ret > 0); n_tot += 1
        print(f'  {yr:<6} {yr_ret:>12.2%} {yr_act:>13d}  {period}')
    print(f'  {"-"*50}')
    print(f'  Positive years: {n_pos}/{n_tot}')
    print(f'{"="*58}')
    print()

## 21. Discussion of Results

**ETF screening:** The IS heatmap (Section 15) reveals which sectors best suit each signal type. Trend-following signals (MA Crossover, Donchian) tend to score higher on sectors with persistent multi-year trends (e.g. Technology, Energy), while RSI mean-reversion benefits from range-bound sectors (Utilities, Consumer Staples).

**IS vs OOS decay:** Some IS-to-OOS decay is expected under the efficient market hypothesis. The key empirical test is whether OOS Sortino remains positive — a sign that the signal captures a genuine, persistent effect rather than overfit noise.

**OOS2 (2000–2009):** The hardest stress-test, spanning two of the worst bear markets of the last 30 years (dot-com bust 2000–2002, early GFC 2007–2009). Long-only signals that flip to cash avoid catastrophic drawdowns. Trend-following benefits here more than RSI, which may generate premature buy signals during prolonged downtrends.

**Signal diversification:** MA Crossover and Donchian are both trend-following but differ in mechanism — MA uses exponential level-crossing, Donchian uses price extremes. RSI is a mean-reversion signal, capturing a different regime. Holding all three provides exposure to different market dynamics.

**Neighbourhood test:** Flat Sortino around the IS optimum indicates the edge is not concentrated in a single lucky parameter combination — it spans a neighbourhood, suggesting genuine robustness rather than over-fitting.

## 22. Limitations

1. **Parameter optimisation creates data-snooping risk.** Sorting 10 ETFs × many parameter combos increases the chance of selecting a lucky combination. The Deflated Sharpe Ratio (Bailey & Lopez de Prado, 2014) would correct for this.

2. **Long-only bias.** The strategy sits in cash during bear markets instead of going short. The cash portion earns 0% — in practice it would earn the risk-free rate.

3. **Simplified transaction costs.** 10 bps ignores market impact, bid-ask widening during stress (e.g. March 2020), and ETF borrow costs for any future short extension.

4. **OOS2 is backward in time.** 2000–2009 precedes IS 2010–2019, so it is a pre-sample validation, not a true chronological forward test. A Mincer-Zarnowitz test would be more rigorous.

5. **ETF history limits.** IYR (Real Estate) launched in June 2000, so the first months of OOS2 may have fewer data points. The `len < 100` guard catches this but is a simple heuristic.

6. **IS/OOS split sensitivity.** All results depend on the 2010/2019 boundary. An alternative split (e.g. 2010–2017 IS, 2018–2025 OOS) could yield different rankings.

7. **Single ETF per signal.** Diversifying each signal across multiple sector ETFs could reduce idiosyncratic risk and smooth the equity curve.

## 23. Conclusion

This notebook tested **Moving Average Crossover**, **RSI**, and **Donchian Channel Breakout** strategies across all 10 SPDR/iShares sector ETFs using a strict walk-forward framework with a one-day execution lag and 10 bps one-way transaction costs.

Parameters were optimised exclusively on the in-sample period 2010–2019 and then applied unchanged to two out-of-sample windows — OOS1 2020–2025 (forward) and OOS2 2000–2009 (backward pre-sample stress-test) — spanning four distinct market regimes.

The strongest in-sample strategy and its OOS generalisation are identified by the Min OOS Sortino criterion (Section 16), which is the most conservative robustness measure available without touching out-of-sample data during selection.

Overall, the results suggest that simple breakout and trend-following signals capture a genuine systematic edge in sector ETFs, particularly during trending regimes. Mean-reversion (RSI) is more sensitive to the market environment. All results should be interpreted cautiously given the multiple-testing exposure inherent in a 30-combination search.

---
*References*
- Brock, Lakonishok & LeBaron (1992). Simple Technical Trading Rules. *J. of Finance*, 47(5).
- Wilder, J. W. (1978). *New Concepts in Technical Trading Systems*. Trend Research.
- Sortino & van der Meer (1991). Downside Risk. *J. of Portfolio Management*, 17(4).
- Bailey & Lopez de Prado (2014). The Deflated Sharpe Ratio. *J. of Portfolio Management*, 40(5).
- Faith, C. (2003). *Way of the Turtle*. McGraw-Hill.